## Sistema de Recomendação de Cartões de Crédito

Este notebook tem como objetivo:
1. Desenvolver um modelo preditivo para classificação do principal cartão para clientes
2. Aplicar o modelo de clientes na base prospects
3. Gerar arquivo final com cartão ideal (recomendação) para prospects

### Importar bibliotecas necessárias

In [ ]:
# EDA e Visualização de Dados
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, f_oneway
from colorama import Fore, Back, Style

# Configurar formato de exibição para não usar notação científica
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True, precision=5)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, log_loss
from catboost import CatBoostClassifier, Pool, cv

# Otimização
import optuna

# Utilitários
import joblib
import math

# Abrir Base Clientes

In [ ]:
df_clientes = pd.read_csv('datasets/clientes.csv')

In [ ]:
df_clientes.info()

In [ ]:
# Remover colunas únicas
df_clientes.drop(columns=['ID_Cliente', 'Nome'], axis=1, inplace=True)

In [ ]:
num_vars = df_clientes.select_dtypes(include=['number']).columns
cat_vars = df_clientes.select_dtypes(include=['object']).columns
target = 'Principal Cartão'

# EDA

## Testes de Hipóteses

In [ ]:
# Testes de hipóteses entre Target Categórica e Numéricas (ANOVA)
for num_col in num_vars:
    groups = [df_clientes[df_clientes[target] == val][num_col] for val in df_clientes[target].unique()]
    stat, p = f_oneway(*groups)
    print(f"{Fore.RED if p < 0.05 else Fore.WHITE}"
            f"ANOVA entre {num_col} e {target}: p-valor = {p}")

## Analisando relação entre variáveis explicativas e targets

In [ ]:

for col in num_vars:
    fig = px.box(df_clientes, x=target, y=col, title=f"{col} por {target}")
    fig.show()

for col in cat_vars:
    fig = px.histogram(df_clientes, x=col, color=target, barmode='group', title=f"{col} por {target}")
    fig.show()

# Modelo Catboost com Validação Cruzada

In [ ]:
selected_features = ['Viagens', 'Restaurantes', 'Entretenimento', 'Cashback', 'Compras online', 'Farmácias',
                     'Programas de Milhagem', 'Postos de Combustível', 'Mercados', 'Score']
X = df_clientes[selected_features]
y = df_clientes[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

# Definir os parâmetros do modelo
params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'random_strength': 2,
    'loss_function': 'MultiClass',  # Use 'MultiClass' para problemas de classificação multiclasse
    'eval_metric': 'MultiClass'
}

model = CatBoostClassifier(**params, random_seed=42, auto_class_weights='Balanced')

model.fit(X_train, y_train)
# Obter o log loss médio
best_score = model.best_score_['learn']['MultiClass']
print(f"LogLoss no treinamento: {best_score}")

In [ ]:
y_pred = model.predict(X_test)
probs = model.predict_proba(X_test)
test_loss = log_loss(y_test, probs)
print(f"LogLoss no conjunto de teste: {test_loss:.2f}")

## Otimizar com Optuna

In [ ]:
# Definir a função objetivo para otimização
def objective(trial):
    # Definir o espaço de busca para os hiperparâmetros
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1, 10),
        'random_strength': trial.suggest_uniform('random_strength', 0, 10),
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass'
    }
    
    model = CatBoostClassifier(**params, random_seed=42, auto_class_weights='Balanced')

    model.fit(X_train, y_train)
    best_score = model.best_score_['learn']['MultiClass']

    return best_score

# Criar um estudo e otimizar
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)  # Ajuste o número de trials conforme necessário

# Obter os melhores parâmetros
best_params = study.best_params
best_metric = study.best_value
print("Melhores parâmetros:", best_params)
print("Melhor métrica:", best_metric)

In [ ]:
# Treinar o modelo final com os parâmetros otimizados
best_model = CatBoostClassifier(**best_params,
                               verbose=False,
                               random_seed=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)
test_loss = log_loss(y_test, probs)
print(f"LogLoss no conjunto de teste: {test_loss:.2f}")

In [ ]:
# Calcular e exibir métricas
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

# Criar matriz de confusão
plt.figure(figsize=(10, 8))
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

# Plotar importância das features
plt.figure(figsize=(10, 6))
feature_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.title('Importância das Features')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

## Salvar Modelo

In [ ]:
joblib.dump(best_model, 'modelo_recomendacao.pkl')

# Abrir Base Prospects com Score

In [14]:
# Carregar Prospects
df_prospects = pd.read_csv('./datasets/prospects_com_score.csv')

In [15]:
df_prospects.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID_Prospect                  500 non-null    object 
 1   Nome                         500 non-null    object 
 2   Idade                        500 non-null    int64  
 3   Viagens                      500 non-null    int64  
 4   Restaurantes                 500 non-null    int64  
 5   Entretenimento               500 non-null    int64  
 6   Cashback                     500 non-null    int64  
 7   Compras online               500 non-null    int64  
 8   Farmácias                    500 non-null    int64  
 9   Programas de Milhagem        500 non-null    int64  
 10  Postos de Combustível        500 non-null    int64  
 11  Mercados                     500 non-null    int64  
 12  Cidade                       500 non-null    object 
 13  Cargo               

# Fazer Pedrição usando modelo

## Carregar Modelo

In [16]:
modelo = joblib.load('modelo_recomendacao.pkl')

## Preparar dados

In [17]:
# Preparar dados
selected_features = ['Viagens', 'Restaurantes', 'Entretenimento', 'Cashback', 'Compras online', 'Farmácias',
                     'Programas de Milhagem', 'Postos de Combustível', 'Mercados', 'Score']
X_prospects = df_prospects[selected_features]

## Realizar Predição

In [18]:
# Fazer previsões
df_prospects['Principal Cartão'] = modelo.predict(X_prospects).squeeze()

In [20]:
df_prospects.head(10)

,ID_Prospect,Nome,Idade,Viagens,Restaurantes,Entretenimento,Cashback,Compras online,Farmácias,Programas de Milhagem,Postos de Combustível,Mercados,Cidade,Cargo,Estado Civil,Tempo na Empresa,Pessoas em Casa,Moradia,Possui Carro,Renda,Investimentos,Ativos,Dívidas,Probabilidade Inadimplencia,Score,Principal Cartão
0,de943fb0-5e23-4e7f-8130-1aa76b33f4dc,Francisco Fonseca,24,1,1,4,5,4,1,5,1,2,Brasília,Coach,Divorciado,14,6,Próprio,True,46006,736831,371505,88180,59.53000,225,Rocketseat Mastercard Life Elite
1,5050d0e5-5d35-4cb1-a208-596a3ee4ac1d,Melissa Mendes,58,1,4,5,3,2,3,3,1,2,São Paulo,Especialista em agronegócios,Solteiro,4,2,Próprio,False,38086,374702,1828554,31206,51.75000,277,Rocketseat Visa Life Platinum
2,435005c0-50ea-4755-acef-b4362c08dccd,Lucca Costa,18,2,4,1,1,3,1,3,2,2,Campo Grande,Cobaia Médica,Divorciado,20,6,Próprio,False,42531,933858,1295274,178144,76.90000,140,Rocketseat Visa Life Select
3,4e9951b4-1790-484c-ab25-189aa9089241,Francisco da Luz,20,4,3,1,4,1,4,2,2,4,Manaus,Classificador contábil,Solteiro,11,6,Próprio,True,16112,698799,1574952,434757,84.60000,23,Rocketseat Visa Shopping Basic
4,cb434840-2bf0-4516-8a79-8c9ada489c03,Sr. Vinícius Costela,22,1,1,4,3,5,2,3,1,2,Goiânia,Sapateiro,Solteiro,0,3,Próprio,True,3059,920950,449866,224609,62.81000,62,Rocketseat Mastercard Shopping Basic
5,cbadf676-7690-4837-891d-9f9d7eb20d95,Emanuelly da Mota,20,5,5,3,1,1,5,3,5,1,Porto Alegre,Paginador,Divorciado,19,5,Alugado,False,23682,796881,1572953,127732,81.34000,90,Rocketseat Mastercard Travel Basic
6,2b07d568-893c-4199-8927-b47ce77afeef,Arthur Gabriel da Cruz,76,3,5,3,2,1,3,4,1,1,São Paulo,Digitador,Viúvo,15,5,Alugado,True,27069,598939,898573,494353,71.88000,25,Rocketseat Mastercard Life Basic
7,ca81a1c4-9cc5-4519-a781-8bc6509a6e39,Benicio Barbosa,76,5,1,1,5,4,4,4,3,1,Florianópolis,Metalúrgico,Casado,15,5,Próprio,True,47966,932984,456768,347189,55.73000,191,Rocketseat Visa Life Select
8,c2fa002d-97ca-4abf-b473-098651f7dd58,Danilo Novaes,30,5,3,3,5,3,2,1,1,4,Fortaleza,Engenheiro florestal,Viúvo,12,2,Próprio,False,1527,228554,336970,331813,85.58000,0,Rocketseat Visa Travel Basic
9,f7a15aaf-9b2c-4094-abb7-d7882b9054e0,Rebeca Aragão,40,1,1,4,2,4,3,3,2,4,Belo Horizonte,Lixeiro/Coletor de lixo,Divorciado,3,4,Próprio,True,39149,64118,1316458,97921,77.52000,87,Rocketseat Visa Life Basic


## Salvar Resultados da Recomendação

In [19]:
# Salvar resultados
df_prospects.to_csv('datasets/prospects_com_recomendacao.csv', index=False)